# 🧠 RAG Advanced — 02: Hybrid Search

## The Problem with Pure Semantic Search

Our current retriever uses **semantic search** (embeddings).
It finds chunks that are *conceptually similar* to the query.

But it can miss results when:
- The user uses **exact keywords** ("BLEU score", "GPT-3", "attention heads")
- The answer is in a chunk that's **not semantically close** but has the exact words

## The Solution: Hybrid Search

Combine TWO retrieval methods:

| Method | How | Good at |
|--------|-----|---------|
| **BM25** | Keyword matching (exact words) | Specific terms, names, numbers |
| **FAISS** | Semantic similarity (meaning) | Concepts, paraphrasing |

## What is RRF?
**Reciprocal Rank Fusion** — a formula that merges results from both retrievers into one ranked list.

```
BM25 results:  [chunk3, chunk7, chunk1, ...]
FAISS results: [chunk1, chunk3, chunk9, ...]
      ↓
RRF merges and reranks
      ↓
Final: [chunk3, chunk1, chunk7, ...]
```

In [4]:
!pip install langchain langchain-community langchain-huggingface -q
!pip install faiss-cpu sentence-transformers -q
!pip install langchain-groq rank-bm25 -q
!pip install langchain-classic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1

## Step 1: Load & Split Documents

Same as reranking notebook — we'll use the same 4 AI papers
to keep our experiments consistent and comparable.

In [5]:
# Attention is all you need paper
!wget "https://arxiv.org/pdf/1706.03762" -O attention_is_all_you_need.pdf

# BERT paper 
!wget "https://arxiv.org/pdf/1810.04805" -O bert.pdf

# GPT-3 paper
!wget "https://arxiv.org/pdf/2005.14165" -O gpt3.pdf

# RAG paper itself!
!wget "https://arxiv.org/pdf/2005.11401" -O rag_paper.pdf

--2026-06-05 11:06:10--  https://arxiv.org/pdf/1706.03762
Resolving arxiv.org (arxiv.org)... 151.101.67.42, 151.101.3.42, 151.101.195.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.67.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2215244 (2.1M) [application/pdf]
Saving to: ‘attention_is_all_you_need.pdf’

attention_is_all_yo 100%[===================>]   2.11M  --.-KB/s    in 0.04s   

2026-06-05 11:06:10 (50.4 MB/s) - ‘attention_is_all_you_need.pdf’ saved [2215244/2215244]

--2026-06-05 11:06:10--  https://arxiv.org/pdf/1810.04805
Resolving arxiv.org (arxiv.org)... 151.101.195.42, 151.101.3.42, 151.101.131.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.195.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 775166 (757K) [application/pdf]
Saving to: ‘bert.pdf’

bert.pdf            100%[===================>] 757.00K  --.-KB/s    in 0.008s  

2026-06-05 11:06:10 (98.1 MB/s) - ‘bert.pdf’ saved [775166/775166]

--2026-06-05 

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

# Load all 4 papers
pdfs = [
    "/kaggle/working/attention_is_all_you_need.pdf",
    "/kaggle/working/bert.pdf",
    "/kaggle/working/gpt3.pdf",
    "/kaggle/working/rag_paper.pdf"
]

all_docs = []
for pdf in pdfs:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    for doc in docs:
        doc.metadata["source"] = pdf.split("/")[-1]
    all_docs.extend(docs)
    print(f"✅ {pdf.split('/')[-1]}: {len(docs)} pages")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50
)
chunks = splitter.split_documents(all_docs)

print(f"\nTotal pages: {len(all_docs)}")
print(f"Total chunks: {len(chunks)}")

/tmp/ipykernel_58/734585687.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ attention_is_all_you_need.pdf: 15 pages
✅ bert.pdf: 16 pages
✅ gpt3.pdf: 75 pages
✅ rag_paper.pdf: 19 pages

Total pages: 125
Total chunks: 941


## Step 2: Create Two Separate Retrievers

### Retriever 1: FAISS (Semantic Search)
- Uses embeddings to find conceptually similar chunks
- Good at: meaning, paraphrasing, concepts

### Retriever 2: BM25 (Keyword Search)
- Uses word frequency to find exact keyword matches
- Good at: specific terms, names, numbers, acronyms

### Why two retrievers?
Each one catches what the other misses.
- "What is attention mechanism?" → FAISS wins (concept)
- "What is the BLEU score of 28.4?" → BM25 wins (exact number)

In [7]:
from langchain_community.retrievers import BM25Retriever

# Retriever 1: FAISS (Semantic)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = FAISS.from_documents(chunks, embeddings)
faiss_retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

# Retriever 2: BM25 (Keyword)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

print("✅ FAISS retriever ready — Semantic Search")
print("✅ BM25 retriever ready — Keyword Search")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS retriever ready — Semantic Search
✅ BM25 retriever ready — Keyword Search


## Step 3: RRF — Reciprocal Rank Fusion

Now we need to **merge** results from both retrievers into one ranked list.

### How RRF works:

Each retriever returns results with a rank (1st, 2nd, 3rd...).
RRF gives each chunk a score based on its rank:

```
RRF score = 1 / (k + rank)
```

Where k=60 is a constant that reduces the impact of high ranks.

### Example:
```
BM25:  [chunkA(1st), chunkB(2nd), chunkC(3rd)]
FAISS: [chunkB(1st), chunkD(2nd), chunkA(3rd)]

chunkA score = 1/(60+1) + 1/(60+3) = 0.0163 + 0.0156 = 0.0319
chunkB score = 1/(60+2) + 1/(60+1) = 0.0161 + 0.0163 = 0.0324

Final ranking: [chunkB, chunkA, chunkD, chunkC]
```

The chunk that appears high in BOTH retrievers wins! 🏆

In [8]:
from langchain_classic.retrievers import EnsembleRetriever

# RRF — Combine both retrievers
# weights = how much to trust each retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.5, 0.5]  # Equal weight to both
)

print("✅ Hybrid retriever ready!")
print("BM25 weight: 50% | FAISS weight: 50%")
print("Fusion method: Reciprocal Rank Fusion (RRF)")

✅ Hybrid retriever ready!
BM25 weight: 50% | FAISS weight: 50%
Fusion method: Reciprocal Rank Fusion (RRF)


## Step 4: Build RAG Chain with Hybrid Retriever

Now we connect the hybrid retriever to the LLM.
Same chain structure as before — only the retriever changed.

In [9]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["GROQ_API_KEY"] = user_secrets.get_secret("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.3,
    max_tokens=512
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Use the context to answer. If you don't know, say you don't know.\n\nContext:\n{context}"),
    ("human", "{input}"),
])

# Three chains to compare
faiss_chain = create_retrieval_chain(
    faiss_retriever,
    create_stuff_documents_chain(llm, prompt)
)

bm25_chain = create_retrieval_chain(
    bm25_retriever,
    create_stuff_documents_chain(llm, prompt)
)

hybrid_chain = create_retrieval_chain(
    hybrid_retriever,
    create_stuff_documents_chain(llm, prompt)
)

print("✅ All 3 chains ready!")
print("1. FAISS only (semantic)")
print("2. BM25 only (keyword)")
print("3. Hybrid (BM25 + FAISS + RRF)")

✅ All 3 chains ready!
1. FAISS only (semantic)
2. BM25 only (keyword)
3. Hybrid (BM25 + FAISS + RRF)


## Step 5: Compare All 3 Retrievers

We'll test with 2 types of questions:

1. **Keyword question** → BM25 should win
2. **Semantic question** → FAISS should win
3. **Hybrid** → should win both or match the best

Let's see!

In [10]:
def compare_retrievers(question):
    print(f"\n{'='*60}")
    print(f"Question: {question}")
    print(f"{'='*60}")

    # FAISS
    faiss_result = faiss_chain.invoke({"input": question})
    print(f"\n🔵 FAISS (Semantic):")
    print(faiss_result['answer'])
    print(f"Sources: {[d.metadata['source'] for d in faiss_result['context']]}")

    # BM25
    bm25_result = bm25_chain.invoke({"input": question})
    print(f"\n🟡 BM25 (Keyword):")
    print(bm25_result['answer'])
    print(f"Sources: {[d.metadata['source'] for d in bm25_result['context']]}")

    # Hybrid
    hybrid_result = hybrid_chain.invoke({"input": question})
    print(f"\n🟢 Hybrid (BM25 + FAISS + RRF):")
    print(hybrid_result['answer'])
    print(f"Sources: {[d.metadata['source'] for d in hybrid_result['context']]}")

# Test 1 — Keyword question (exact number)
compare_retrievers("What is the BLEU score of the Transformer on English to German translation?")


Question: What is the BLEU score of the Transformer on English to German translation?

🔵 FAISS (Semantic):
The BLEU score of the Transformer on English to German translation is not explicitly mentioned in the given text. However, it is mentioned that the Transformer achieves better BLEU scores than previous state-of-the-art models on the English-to-German and English-to-French newstest2014 tests.
Sources: ['attention_is_all_you_need.pdf', 'attention_is_all_you_need.pdf', 'attention_is_all_you_need.pdf', 'attention_is_all_you_need.pdf', 'attention_is_all_you_need.pdf']

🟡 BM25 (Keyword):
I don't know. The text does not mention the BLEU score of the Transformer on English to German translation.
Sources: ['attention_is_all_you_need.pdf', 'attention_is_all_you_need.pdf', 'attention_is_all_you_need.pdf', 'gpt3.pdf', 'gpt3.pdf']

🟢 Hybrid (BM25 + FAISS + RRF):
The BLEU score of the Transformer on English to German translation is 25.8.
Sources: ['attention_is_all_you_need.pdf', 'attention_is

## Analysis: Hybrid Search Wins!

| Retriever | Answer | Correct? |
|-----------|--------|---------|
| FAISS | "not explicitly mentioned" | ❌ |
| BM25 | "I don't know" | ❌ |
| Hybrid | **25.8 BLEU score** | ✅ |

### What happened?

**FAISS alone:** Found semantically similar chunks about BLEU scores
but missed the exact chunk with the number.

**BM25 alone:** Found chunks with keyword "BLEU" but from wrong context,
even pulled from gpt3.pdf which is irrelevant!

**Hybrid (RRF):** Combined both signals:
- BM25 found chunks with exact keyword "BLEU score"
- FAISS found chunks about transformer performance
- RRF merged them → the chunk with "28.4" ranked highest

### Key Lesson:
> Neither semantic nor keyword search alone is enough.
> Hybrid search finds what both miss individually.

### Real production systems use hybrid because:
- ✅ Handles exact terms (model names, numbers, acronyms)
- ✅ Handles conceptual questions (meaning, paraphrasing)
- ✅ More robust across different question types

In [11]:
# Test 2 — Semantic question (concept)
compare_retrievers("How does self-attention help the model understand context?")


Question: How does self-attention help the model understand context?

🔵 FAISS (Semantic):
I don't know. The text does not explicitly explain how self-attention helps the model understand context. However, it does mention that larger models make increasingly efficient use of in-context information (Figure 1.2), which suggests that self-attention may play a role in this process.
Sources: ['attention_is_all_you_need.pdf', 'rag_paper.pdf', 'attention_is_all_you_need.pdf', 'attention_is_all_you_need.pdf', 'gpt3.pdf']

🟡 BM25 (Keyword):
Self-attention is a mechanism in the Transformer model that helps the model understand context by allowing it to weigh the importance of different input elements relative to each other. In a self-attention layer, all the keys, values, and queries come from the input sequence itself.

Here's how it works:

1. **Keys, Values, and Queries**: The input sequence is split into three components: keys, values, and queries. The keys and values are used to compute the

## Analysis: Semantic Question Results

| Retriever | Answer Quality |
|-----------|---------------|
| FAISS | ❌ "I don't know" — missed completely |
| BM25 | ✅ Detailed and accurate |
| Hybrid | ✅ Good but less detailed than BM25 |

### Surprise! BM25 won this round!

### Why?
The question contains strong keywords:
"self-attention", "model", "context", "understand"

These exact words appear heavily in the papers →
BM25 keyword matching found the right chunks directly.

### When does FAISS truly win?
When the question is phrased differently from the document.

Example:
- Document says: "multi-head attention allows parallel processing"
- Question asks: "how does the transformer handle multiple things at once?"

These don't share keywords → BM25 fails → FAISS wins.

### Final Verdict on Hybrid Search:

| Question Type | Best Retriever |
|--------------|---------------|
| Exact numbers/names | BM25 ✅ |
| Technical keywords | BM25 ✅ |
| Paraphrased concepts | FAISS ✅ |
| Mixed/Unknown | Hybrid ✅ |

> Hybrid is the safe default — it never performs worse than
> the worst of the two, and often beats both individually.